# Phase 3: Data Cleaning & Feature Engineering

## Objective
Prepare the raw dataset for predictive modeling by resolving data quality issues,
encoding categorical variables, and engineering features that capture business-relevant patterns
discovered during EDA.

## Key questions driving this phase:
- Which columns need type corrections?
- Which categorical variables need encoding, and what method is appropriate for each?
- What new features can we derive that a model cannot learn on its own?

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load raw data — never modify this file
df = pd.read_csv('/content/drive/MyDrive/projects /customer churn analysis/Data/raw/Telco-Customer-Churn.csv')

print("Shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nSample:\n", df.head(3))

Shape: (7043, 21)

Data types:
 customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Sample:
    customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   

      MultipleLines InternetService

## 3.1 Data Quality Audit

Before cleaning, we revisit the known issues flagged during EDA:
- `TotalCharges` is stored as object dtype but should be numeric
- Blank strings in `TotalCharges` represent new customers (tenure = 0) — not true nulls
- `customerID` carries no predictive value and should be dropped
- `Churn` (target variable) must be encoded as binary integer (0/1)

In [3]:
# make a copy to avoid changes in raw file
df_clean= df.copy()
#fix=1 (change dtype of 'monthly charges' from object to numeric)
df_clean= df_clean
df_clean['TotalCharges']= pd.to_numeric(df_clean['TotalCharges'],errors='coerce')
#fix 2 (these nan values in monthly charges are showing new customers with tenure=0,fill nan with 0)
df_clean['TotalCharges']=df_clean['TotalCharges'].fillna(0)
# 3. Drop customerID — it's an identifier, not a predictor
df_clean.drop(columns=['customerID'], inplace=True)

# 4. Encode target variable
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

# Verify
print("TotalCharges dtype:", df_clean['TotalCharges'].dtype)
print("Churn value counts:\n", df_clean['Churn'].value_counts())
print("Remaining nulls:\n", df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

TotalCharges dtype: float64
Churn value counts:
 Churn
0    5174
1    1869
Name: count, dtype: int64
Remaining nulls:
 Series([], dtype: int64)


## 3.2 Feature Engineering

### New feature: `early_tenure_flag`
EDA revealed that customers in their first 12 months have significantly higher churn rates.
This binary flag encodes that business insight directly into the dataset, making it
explicitly available to the model rather than leaving it to infer from raw tenure values alone.

In [4]:
# Flag customers in their first year
df_clean['early_tenure_flag'] = (df_clean['tenure'] <= 12).astype(int)

# Validate — churn rate should be notably higher for flag=1
validation = df_clean.groupby('early_tenure_flag')['Churn'].mean().round(3)
print("Churn rate by early tenure flag:")
print(validation)
print("\nFlag distribution:")
print(df_clean['early_tenure_flag'].value_counts())

Churn rate by early tenure flag:
early_tenure_flag
0    0.171
1    0.474
Name: Churn, dtype: float64

Flag distribution:
early_tenure_flag
0    4857
1    2186
Name: count, dtype: int64


## 3.3 Encoding Categorical Variables

Binary categorical columns (two unique values) are label encoded (0/1).
Multi-category columns are one-hot encoded to avoid implying false ordinal relationships.
The target variable `Churn` was already encoded in Step 3.1.

In [6]:
df_clean.shape


(7043, 21)

In [7]:
# ── 3.3 Encoding Categorical Variables ──────────────────────────────────────

# Separate binary and multi-category columns
binary_cols = [col for col in df_clean.select_dtypes(include='object').columns
               if df_clean[col].nunique() == 2]

multi_cols = [col for col in df_clean.select_dtypes(include='object').columns
              if df_clean[col].nunique() > 2]

print("Binary columns  →  Label Encode:", binary_cols)
print("Multi  columns  →  One-Hot Encode:", multi_cols)

# --- Label Encode binary columns ---
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in binary_cols:
    df_clean[col] = le.fit_transform(df_clean[col])
    print(f"  {col}: {le.classes_} → [0, 1]")

# --- One-Hot Encode multi-category columns ---
df_clean = pd.get_dummies(df_clean, columns=multi_cols, drop_first=True)

print(f"\n Encoding complete. Dataset shape: {df_clean.shape}")
print(df_clean.head(3))

Binary columns  →  Label Encode: ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
Multi  columns  →  One-Hot Encode: ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']
  gender: ['Female' 'Male'] → [0, 1]
  Partner: ['No' 'Yes'] → [0, 1]
  Dependents: ['No' 'Yes'] → [0, 1]
  PhoneService: ['No' 'Yes'] → [0, 1]
  PaperlessBilling: ['No' 'Yes'] → [0, 1]

 Encoding complete. Dataset shape: (7043, 32)
   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   

   PaperlessBilling  MonthlyCharges  TotalCharges  Churn  ...  \
0                 1           29.85         29.85      0  ...   
1                 0           56.95       1

After encoding, the dataset expanded from 21 columns to 32 columns due to one-hot encoding.
All features are now numeric and model-ready. No ordinal assumptions were imposed on
multi-category variables.

In [8]:
# ── 3.4 Feature Engineering ──────────────────────────────────────────────────

# --- Feature 1: Tenure Group ---
# EDA showed churn is highest in the first 12 months.
# Bucketing tenure makes this non-linear relationship explicit for the model.

def tenure_group(tenure):
    if tenure <= 12:
        return 'New'        # Highest risk
    elif tenure <= 36:
        return 'Developing' # Medium risk
    else:
        return 'Loyal'      # Lowest risk

df_clean['tenure_group'] = df_clean['tenure'].apply(tenure_group)

# Ordinal encode — order matters here (New < Developing < Loyal)
tenure_map = {'New': 0, 'Developing': 1, 'Loyal': 2}
df_clean['tenure_group'] = df_clean['tenure_group'].map(tenure_map)

print("✅ Feature 1: tenure_group created")
print(df_clean['tenure_group'].value_counts())


# --- Feature 2: Number of Services ---
# More services = higher switching cost = lower churn likelihood.
# We sum all service-related binary columns (already encoded as 0/1).

service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies'
]

# Only include columns that exist (after one-hot, some names may have changed)
existing_service_cols = [col for col in service_cols if col in df_clean.columns]
df_clean['num_services'] = df_clean[existing_service_cols].sum(axis=1)

print("\n✅ Feature 2: num_services created")
print(df_clean['num_services'].value_counts().sort_index())


# --- Feature 3: Average Monthly Charge per Service ---
# High cost relative to services used may signal poor value perception → churn risk.

df_clean['charge_per_service'] = (
    df_clean['MonthlyCharges'] / (df_clean['num_services'] + 1)  # +1 avoids division by zero
)

print("\n✅ Feature 3: charge_per_service created")
print(df_clean['charge_per_service'].describe())


print(f"\n📐 Final dataset shape: {df_clean.shape}")

✅ Feature 1: tenure_group created
tenure_group
2    3001
0    2186
1    1856
Name: count, dtype: int64

✅ Feature 2: num_services created
num_services
0     682
1    6361
Name: count, dtype: int64

✅ Feature 3: charge_per_service created
count    7043.000000
mean       34.415739
std        15.105349
min         9.125000
25%        24.600000
50%        37.550000
75%        46.337500
max        67.200000
Name: charge_per_service, dtype: float64

📐 Final dataset shape: (7043, 35)


Three business-driven features were engineered from EDA insights:

1.`tenure_group` captures the non-linear churn risk across customer lifecycle stages.

2.`num_services` proxies switching cost and customer stickiness.

3.`charge_per_service`
flags potential value dissatisfaction. Each feature is directly traceable to a
Phase 2 finding.

In [9]:
# ── 3.5 Preparing the Final Model-Ready Dataset ─────────────────────────────

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Step 1: Separate Features and Target ---
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

print(f"Features shape : {X.shape}")
print(f"Target shape   : {y.shape}")
print(f"Churn rate in full dataset: {y.mean():.2%}")


# --- Step 2: Train-Test Split (80/20) ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,       # reproducibility
    stratify=y             # preserves churn ratio in both splits
)

print(f"\n✅ Split complete")
print(f"   Train size : {X_train.shape[0]} rows | Churn rate: {y_train.mean():.2%}")
print(f"   Test size  : {X_test.shape[0]} rows  | Churn rate: {y_test.mean():.2%}")


# --- Step 3: Scale Numeric Features ---
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
                'num_services', 'charge_per_service']

# Only scale columns that exist
numeric_cols = [col for col in numeric_cols if col in X_train.columns]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])  # fit + transform on train
X_test[numeric_cols]  = scaler.transform(X_test[numeric_cols])       # transform only on test

print(f"\n✅ Scaling complete on: {numeric_cols}")


# --- Step 4: Save Processed Data ---
X_train.to_csv('/content/drive/MyDrive/projects /customer churn analysis/Data/processed/X_train.csv', index=False)
X_test.to_csv('/content/drive/MyDrive/projects /customer churn analysis/Data/processed/X_test.csv', index=False)
y_train.to_csv('/content/drive/MyDrive/projects /customer churn analysis/Data/processed/y_train.csv', index=False)
y_test.to_csv('/content/drive/MyDrive/projects /customer churn analysis/Data/processed/y_test.csv', index=False)

print("\n✅ Processed data saved to data/processed/")
print(f"\n📐 Final feature count going into Phase 4: {X_train.shape[1]}")

Features shape : (7043, 34)
Target shape   : (7043,)
Churn rate in full dataset: 26.54%

✅ Split complete
   Train size : 5634 rows | Churn rate: 26.54%
   Test size  : 1409 rows  | Churn rate: 26.54%

✅ Scaling complete on: ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_services', 'charge_per_service']

✅ Processed data saved to data/processed/

📐 Final feature count going into Phase 4: 34


The final model-ready dataset contains 34 features after encoding and engineering.
An 80/20 stratified split preserves the 26.5% churn rate in both train and test sets.
Numeric features are standardized using StandardScaler fitted only on training data
to prevent data leakage. Processed files are saved for direct use in Phase 4.